# Car Price Prediction project (EDA)

In this notebook, we'd like to perform exploratory data analysis *on train set only*,
to prevent data leakage and to comply with ML best practices. We'll try to keep our data
analysis constrained and useful.

## Environment setup

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

## Dataset overview

In [53]:
path = "../src/price_prediction/data/prepared/train.csv"
df = pd.read_csv(path)
df.head()

,Price,Levy,Manufacturer,Model,Prod. year,Category,Leather interior,Fuel type,Engine volume,Mileage,Cylinders,Gear box type,Drive wheels,Doors,Wheel,Color,Airbags
0,10036,-,LINCOLN,Navigator,1998,Jeep,Yes,Petrol,5.4,155200 km,8.0,Automatic,4x4,04-May,Left wheel,Black,6
1,1882,749,KIA,Optima,2014,Sedan,Yes,Petrol,2.4,206211 km,4.0,Automatic,Front,04-May,Left wheel,Blue,12
2,1700,-,VOLKSWAGEN,Golf,1997,Hatchback,No,Petrol,1.8,1000 km,4.0,Manual,Front,04-May,Left wheel,Green,2
3,28852,-,TOYOTA,Highlander,2008,Jeep,No,Hybrid,3.3,249600 km,6.0,Variator,4x4,04-May,Left wheel,Silver,12
4,1882,456,TOYOTA,Camry,2015,Sedan,Yes,Hybrid,2.5,273651 km,4.0,Automatic,Front,04-May,Left wheel,Black,12


In [ ]:
df.info()

In [ ]:
df.select_dtypes(["float64", "int64"]).describe()

In [ ]:
df.duplicated().sum()

In [ ]:
features = df.columns.drop("Price")
df.duplicated(subset=list(features)).sum()

## Target analysis

### Distribution (capped)

In [ ]:
df["Price"].describe()

In [ ]:
perc_99 = np.percentile(df["Price"], [99]) # Clipped due to extremely high prices (> 26,000,000 USD)
df["Price"].hist(bins=50, range=(0, perc_99[0]))
plt.show()

In [ ]:
# Because target distribution is right-skewed with a very long tail,
# check distribution after log transform
log_price = np.log(df["Price"])
log_price.hist(bins=50)
plt.show()

Log transforming target leads to a distribution much closer to normal, so it's definitely an option.

### Skewness

In [ ]:
skew_val = round(df["Price"].skew(), 4)
print(f"target skew : {skew_val}")

Such a large skewness further justifies log transformation, which can significantly help and
lead to more stable learning in linear models.

### Outliers

In [ ]:
quartiles = np.percentile(log_price, [25, 75])
iqr = round(quartiles[1] - quartiles[0], 6)

lower = round(quartiles[0] - 1.5 * iqr, 6)
upper = round(quartiles[1] + 1.5 * iqr, 6)

print("Outlier statistics for target :")
print(f"Price (IQR) : {iqr}")
print(f"Price (Lower Bound) : {lower}")
print(f"Price (Upper Bound) : {upper}")

In [ ]:
too_low = log_price.loc[log_price < lower]
too_high = log_price.loc[log_price > upper]

print(f"Lower outliers (count) : {len(too_low)}")
print(f"Higher outliers (count) : {len(too_high)}")

In [ ]:
sns.boxplot(df["Price"], log_scale=True, orient="v")

Based on the above analysis, performing log transformation on target values is justified, because :
- Using original prices with a very high skewness (> 94) will hurt performance of linear models.
- Log transformed prices with relatively low outlier ratio (~4.4%) lead to more stable training, especially in linear models.

## Feature quality

Let's review what we found during initial inspection of raw data :
- Some categorical features are actually numerical and need some feature engineering :
  - Mileage : Values are encoded as "(distance) km". The suffix " km" can be stripped in a custom transform.
  - Engine volume : Values are encoded as "(volume) Turbo" or "(volume)". Can be split into two features : "Engine volume" (float) and "Turbo" (0/1).
  - Levy : Values are either a number or the missing value indicator "-".
- Several features have high cardinality (>= 5) : Category, Color, Fuel type (and possibly, Manufacturer and Model)
- Some features have 2 unique values and may be encoded as 0/1 (Leather interior and Wheel)
- Remaining categoricals have low cardinality and can be one-hot encoded (Doors, Gear box type and Drive wheels)

### Cardinality of categorical features

In [ ]:
cat_features = [
    "Category", "Color", "Fuel type", "Manufacturer", "Model",
    "Leather interior", "Wheel", "Doors", "Gear box type", "Drive wheels"
]

val_counts = []
for col in cat_features:
    val_counts.append(pd.Series({
        "Name": col,
        "Cardinality": df[col].nunique()
    }))
df_counts = pd.concat(val_counts, axis=1).T
df_counts

In [ ]:
levy_vals = df["Levy"].loc[df["Levy"] != "-"]
levy_vals.astype(np.float64).describe()

In [ ]:
mileage_vals = df["Mileage"].map(lambda x: x.replace("km", "").strip())
mileage_vals.astype(np.float64).describe()

### Correlation of numerical features

In [ ]:
# Some temporary transforms to get a more complete correlation analysis
df_copy = df.copy()
levy_mean = df_copy["Levy"].loc[df_copy["Levy"] != "-"].astype(np.float64).mean()

df_copy["Levy"] = df_copy["Levy"].map(lambda x: levy_mean if x == "-" else x).astype(np.float64)
df_copy["Mileage"] = df_copy["Mileage"].map(lambda x: float(x.replace(" km", "")))
df_copy["Turbo"] = df_copy["Engine volume"].map(lambda x: 1 if x.endswith(" Turbo") else 0).astype(np.int64)
df_copy["Engine volume"] = df_copy["Engine volume"].map(lambda x: x.replace(" Turbo", "")).astype(np.float64)

df_copy.head()

In [ ]:
X = df_copy.drop("Price", axis=1)
df_corr = X.corr(numeric_only=True)
df_corr

## Feature-target relationships

In [ ]:
df_copy["Log Price"] = np.log(df_copy["Price"])
df_copy["Log Mileage"] = np.log(df_copy["Mileage"])
df_log = df_copy.drop(["Price", "Mileage"], axis=1)

sns.pairplot(df_log)

In [ ]:
df_corr = df_log.corr(numeric_only=True)
df_corr

## EDA summary
Based on the above analysis, the following stages are required inside preprocessing pipeline :

### Data cleaning
- All numeric values will be cast as floating point values.

### Feature selection
- Feature "Model" will be dropped due to very high cardinality (n = 1180).

### Imputation
- All categorical features will be imputed using "most frequent" (mode) strategy.
- All numeric features will be imputed using median values.

### Feature engineering
- "Levy" and "Mileage" will be transformed to numeric features.
- "Engine volume" will be split to a numeric feature (Engine volume) and a binary feature (Turbo).
- "Manufacturer" also has high cardinality (n = 60) and brand names will be mapped to country names. Mapped values will be one-hot encoded.
- "Mileage" will be log transformed due to very high variance and extreme values.

### Encoding
- Features with 2 unique values (Leather interior and Wheel) will be encoded as binary (0/1) values.
- Features with 3 or 4 unique values ("Doors", "Drive wheels" and "Gear box type") will be one-hot encoded.
- Features with more uniques values (Category, Color and Fuel type) will also be one-hot encoded, since we have enough training data to learn from some more (6 + 11 + 16) sparse features.

### Numeric scaling
- All numeric features will be scaled using StandardScaler.

### Target transformation
- Target (Price) will also be log transformed due to highly skewed distribution.

## Notes

One rare Fuel type category was removed as part of data cleaning; this is expected and
will be handled via unknown-category tolerance in encoding.

## Pipeline smoke test

In [2]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import (
    OneHotEncoder, StandardScaler, FunctionTransformer)

In [3]:
NUM_FEATURES = ["Prod. year", "Cylinders", "Airbags"]
CAT_FEATURES = [
    "Doors", "Drive wheels", "Gear box type",
    "Category", "Color", "Fuel type"
]
BIN_FEATURES = ["Leather interior", "Wheel"]

In [4]:
_country_manufacturer_map = {
  "United States": [
      "LINCOLN", "CHEVROLET", "FORD", "DODGE", "CHRYSLER", "JEEP",
      "CADILLAC", "SCION", "HUMMER", "BUICK", "GMC", "SATURN",
      "MERCURY", "TESLA", "PONTIAC"
  ],
  "South Korea": ["KIA", "HYUNDAI", "SSANGYONG", "DAEWOO"],
  "Germany": [
      "VOLKSWAGEN", "MERCEDES-BENZ", "AUDI", "BMW", "OPEL", "PORSCHE"
  ],
  "Japan": [
      "TOYOTA", "ACURA", "HONDA", "LEXUS", "MAZDA", "NISSAN", "SUBARU",
      "MITSUBISHI", "SUZUKI", "INFINITI", "DAIHATSU", "ISUZU"
  ],
  "Russia": ["VAZ", "GAZ", "MOSKVICH", "UAZ"],
  "Italy": [
      "ALFA ROMEO", "MASERATI", "FIAT", "FERRARI", "LAMBORGHINI"
  ],
  "France": ["RENAULT", "CITROEN", "PEUGEOT"],
  "Czech Republic": ["SKODA"],
  "Ukraine": ["ZAZ"],
  "United Kingdom": [
      "MINI", "JAGUAR", "LAND ROVER", "ROVER", "ASTON MARTIN"
  ],
  "Sweden": ["VOLVO", "SAAB"],
  "China": ["GREATWALL"],
  "Spain": ["SEAT"]
}


def get_country_map():
    country_map = {}
    for country in _country_manufacturer_map:
        mapping = { manufacturer: country
                    for manufacturer in _country_manufacturer_map[country] }
        country_map.update(mapping)

    return country_map

In [26]:
def get_values(x):
    if isinstance(x, (pd.DataFrame, pd.Series)):
        values = x.values.flatten()
    else:
        values = x.flatten()
    return values


def manufacturer_transform(x):
    country_map = get_country_map()
    countries = []
    for m in x:
        key = m[0] if isinstance(m, (list, np.ndarray)) else m
        country = country_map[key] if key in country_map else "Unknown"
        countries.append(country)
    return np.array(countries).reshape(-1, 1)


def levy_transform(x):
    values = get_values(x)
    return np.array([
        np.nan if levy == "-" else float(levy) for levy in values
    ]).reshape(-1, 1)


def mileage_transform(x):
    values = get_values(x)
    return np.array([
        float(mileage.replace(" km", "")) for mileage in values
    ]).reshape(-1, 1)


def engine_volume_transform(x):
    values = get_values(x)
    volume_turbo = []
    for engine_volume in values:
        turbo = int(engine_volume.endswith(" Turbo"))
        volume = float(engine_volume.replace(" Turbo", ""))
        volume_turbo.append([volume, turbo])
    return np.array(volume_turbo)


def engine_volume_features(
        transform: FunctionTransformer, input_features=None
):
    _ = transform
    _ = input_features
    return ["Engine volume", "Turbo"]


def cast_to_int(x):
    return x.astype(np.int64)


def build_pipeline() -> ColumnTransformer:
    cat_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
    ])
    num_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ])
    bin_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(
            handle_unknown="ignore", drop="if_binary"
        ))
    ])

    # Custom transforms
    levy_pipeline = Pipeline([
        ("transformer", FunctionTransformer(
            func=levy_transform,
            feature_names_out="one-to-one"
        )),
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ])
    mileage_pipeline = Pipeline([
        ("transformer", FunctionTransformer(
            func=mileage_transform,
            feature_names_out="one-to-one"
        )),
        ("imputer", SimpleImputer(strategy="median")),
        ("log_scaler", FunctionTransformer(func=np.log1p, feature_names_out="one-to-one")),
        ("scaler", StandardScaler())
    ])
    manufacturer_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("mapper", FunctionTransformer(
            func=manufacturer_transform,
            feature_names_out="one-to-one"
        )),
        ("encoder", OneHotEncoder(handle_unknown="ignore"))
    ])
    engine_volume_num_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ])
    turbo_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("cast", FunctionTransformer(
            func=cast_to_int,
            feature_names_out="one-to-one"
        )),
    ])
    engine_volume_splitter = FunctionTransformer(
        func=engine_volume_transform,
        feature_names_out=engine_volume_features
    )
    engine_volume_pipeline = Pipeline([
        ("split", engine_volume_splitter),
        ("postprocess", ColumnTransformer(
            transformers=[
                ("engine_volume", engine_volume_num_pipeline, [0]),
                ("turbo", turbo_pipeline, [1])
            ],
            remainder="drop")
        )
    ])

    return ColumnTransformer([
        ("categorical", cat_pipeline, CAT_FEATURES),
        ("numerical", num_pipeline, NUM_FEATURES),
        ("binary", bin_pipeline, BIN_FEATURES),
        ("levy", levy_pipeline, ["Levy"]),
        ("mileage", mileage_pipeline, ["Mileage"]),
        ("country", manufacturer_pipeline, ["Manufacturer"]),
        ("engine_volume", engine_volume_pipeline, ["Engine volume"])
    ])

In [27]:
path = "../src/price_prediction/data/prepared/train.csv"
data = pd.read_csv(path)
X = data.drop("Price", axis=1)
pipeline = build_pipeline()

In [7]:
X_trans = pipeline.fit_transform(X)
X_trans = pd.DataFrame(X_trans, columns=pipeline.get_feature_names_out())
X_trans.head()

,categorical__Doors_02-Mar,categorical__Doors_04-May,categorical__Doors_>5,categorical__Drive wheels_4x4,categorical__Drive wheels_Front,categorical__Drive wheels_Rear,categorical__Gear box type_Automatic,categorical__Gear box type_Manual,categorical__Gear box type_Tiptronic,categorical__Gear box type_Variator,...,country__Manufacturer_Japan,country__Manufacturer_Russia,country__Manufacturer_South Korea,country__Manufacturer_Spain,country__Manufacturer_Sweden,country__Manufacturer_Ukraine,country__Manufacturer_United Kingdom,country__Manufacturer_United States,engine_volume__engine_volume__Engine volume,engine_volume__turbo__Turbo
0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,4.032517,0.0
1,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.264591,0.0
2,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-0.488995,0.0
3,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.394969,0.0
4,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.390188,0.0


## Pipeline sanity checks

### Shape invariance

In [28]:
X_sample = X.head()
X_trans = pd.DataFrame(pipeline.fit_transform(X), columns=pipeline.get_feature_names_out())
X_sample_trans = pd.DataFrame(pipeline.transform(X_sample), columns=pipeline.get_feature_names_out())

In [29]:
print(f"Sample data : {X_sample_trans.shape}\tFull data : {X_trans.shape}\tRaw data : {X.shape}\n")

Sample data : (5, 65)	Full data : (9001, 65)	Raw data : (9001, 16)



In [31]:
nan_items = np.isnan(X_trans).sum()
inf_items = np.isinf(X_trans).sum()

inf_items.loc[inf_items > 0]

Series([], dtype: int64)

In [32]:
features = pipeline.get_feature_names_out()
features

array(['categorical__Doors_02-Mar', 'categorical__Doors_04-May',
       'categorical__Doors_>5', 'categorical__Drive wheels_4x4',
       'categorical__Drive wheels_Front',
       'categorical__Drive wheels_Rear',
       'categorical__Gear box type_Automatic',
       'categorical__Gear box type_Manual',
       'categorical__Gear box type_Tiptronic',
       'categorical__Gear box type_Variator',
       'categorical__Category_Cabriolet', 'categorical__Category_Coupe',
       'categorical__Category_Goods wagon',
       'categorical__Category_Hatchback', 'categorical__Category_Jeep',
       'categorical__Category_Limousine',
       'categorical__Category_Microbus', 'categorical__Category_Minivan',
       'categorical__Category_Pickup', 'categorical__Category_Sedan',
       'categorical__Category_Universal', 'categorical__Color_Beige',
       'categorical__Color_Black', 'categorical__Color_Blue',
       'categorical__Color_Brown', 'categorical__Color_Carnelian red',
       'categorical__Colo

### Semantic sanity check

**Check 1 : Binary features must be binary**

In [33]:
print("Checking binary features...\n")
print(f"binary__Leather interior_Yes (values) : {X_trans["binary__Leather interior_Yes"].unique()}")
print(f"binary__Wheel_Right-hand drive (values) : {X_trans["binary__Wheel_Right-hand drive"].unique()}")
print(f"engine_volume__turbo__Turbo (values) : {X_trans["engine_volume__turbo__Turbo"].unique()}")

Checking binary features...

binary__Leather interior_Yes (values) : [1. 0.]
binary__Wheel_Right-hand drive (values) : [0. 1.]
engine_volume__turbo__Turbo (values) : [0. 1.]


**Check 2 : One-hot encoded features must behave correctly**

In [34]:
cat_columns = [col for col in X_trans.columns if col.startswith("categorical__Category")]
df_category = X_trans[cat_columns]
df_category.head(10)

,categorical__Category_Cabriolet,categorical__Category_Coupe,categorical__Category_Goods wagon,categorical__Category_Hatchback,categorical__Category_Jeep,categorical__Category_Limousine,categorical__Category_Microbus,categorical__Category_Minivan,categorical__Category_Pickup,categorical__Category_Sedan,categorical__Category_Universal
0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
2,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
6,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
7,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
8,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
9,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


**Check 3 : Log-transformed features are sane**

In [35]:
X_trans["mileage__Mileage"].head()

0    0.340534
1    0.461913
2   -1.813714
3    0.543474
4    0.582766
Name: mileage__Mileage, dtype: float64

### Inference robustness check

In [36]:
X["Fuel type"].unique()

array(['Petrol', 'Hybrid', 'CNG', 'Diesel', 'LPG', 'Plug-in Hybrid'],
      dtype=object)

In [37]:
color = "Magenta"
fuel = "Hydrogen"
manufacturer = "INVALID"

unseen = X.iloc[0, :].copy()
unseen["Color"] = color
unseen["Fuel type"] = fuel
unseen["Manufacturer"] = manufacturer
unseen

Levy                         -
Manufacturer           INVALID
Model                Navigator
Prod. year                1998
Category                  Jeep
Leather interior           Yes
Fuel type             Hydrogen
Engine volume              5.4
Mileage              155200 km
Cylinders                  8.0
Gear box type        Automatic
Drive wheels               4x4
Doors                   04-May
Wheel               Left wheel
Color                  Magenta
Airbags                      6
Name: 0, dtype: object

In [39]:
X_unseen = pd.DataFrame([unseen], columns=X.columns)
X_unseen_trans = pd.DataFrame(pipeline.transform(X_unseen), columns=pipeline.get_feature_names_out())
X_unseen_trans
#X_unseen

,categorical__Doors_02-Mar,categorical__Doors_04-May,categorical__Doors_>5,categorical__Drive wheels_4x4,categorical__Drive wheels_Front,categorical__Drive wheels_Rear,categorical__Gear box type_Automatic,categorical__Gear box type_Manual,categorical__Gear box type_Tiptronic,categorical__Gear box type_Variator,...,country__Manufacturer_Japan,country__Manufacturer_Russia,country__Manufacturer_South Korea,country__Manufacturer_Spain,country__Manufacturer_Sweden,country__Manufacturer_Ukraine,country__Manufacturer_United Kingdom,country__Manufacturer_United States,engine_volume__engine_volume__Engine volume,engine_volume__turbo__Turbo
0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,4.032517,0.0


In [41]:
country_cols = [col for col in X_trans.columns if col.startswith("country__Manufacturer")]
(X_unseen_trans[country_cols] > 0.0).sum()

country__Manufacturer_China             0
country__Manufacturer_Czech Republic    0
country__Manufacturer_France            0
country__Manufacturer_Germany           0
country__Manufacturer_Italy             0
country__Manufacturer_Japan             0
country__Manufacturer_Russia            0
country__Manufacturer_South Korea       0
country__Manufacturer_Spain             0
country__Manufacturer_Sweden            0
country__Manufacturer_Ukraine           0
country__Manufacturer_United Kingdom    0
country__Manufacturer_United States     0
dtype: int64

In [42]:
color_cols = [col for col in X_trans.columns if col.startswith("categorical__Color")]
(X_unseen_trans[color_cols] > 0.0).sum()

categorical__Color_Beige            0
categorical__Color_Black            0
categorical__Color_Blue             0
categorical__Color_Brown            0
categorical__Color_Carnelian red    0
categorical__Color_Golden           0
categorical__Color_Green            0
categorical__Color_Grey             0
categorical__Color_Orange           0
categorical__Color_Pink             0
categorical__Color_Purple           0
categorical__Color_Red              0
categorical__Color_Silver           0
categorical__Color_Sky blue         0
categorical__Color_White            0
categorical__Color_Yellow           0
dtype: int64

In [43]:
fuel_cols = [col for col in X_trans.columns if col.startswith("categorical__Fuel type")]
X_unseen_trans[fuel_cols]

,categorical__Fuel type_CNG,categorical__Fuel type_Diesel,categorical__Fuel type_Hybrid,categorical__Fuel type_LPG,categorical__Fuel type_Petrol,categorical__Fuel type_Plug-in Hybrid
0,0.0,0.0,0.0,0.0,0.0,0.0


### Statistical sanity checks

In [46]:
volume_col = "engine_volume__engine_volume__Engine volume"
levy_col = "levy__Levy"
year_col = "numerical__Prod. year"
cylinder_col = "numerical__Cylinders"
airbag_col = "numerical__Airbags"

X_trans[[volume_col, levy_col, year_col, cylinder_col, airbag_col]].describe()

,engine_volume__engine_volume__Engine volume,levy__Levy,numerical__Prod. year,numerical__Cylinders,numerical__Airbags
count,9.001000e+03,9.001000e+03,9.001000e+03,9.001000e+03,9.001000e+03
mean,-1.042014e-16,-3.552319e-17,1.854547e-14,-8.051923e-17,-7.775632e-17
std,1.000056e+00,1.000056e+00,1.000056e+00,1.000056e+00,1.000056e+00
min,-2.749751e+00,-2.027961e+00,-9.494892e+00,-3.117430e+00,-1.629646e+00
25%,-7.401897e-01,-2.296751e-01,-3.874730e-01,-4.097211e-01,-6.064564e-01
50%,-2.377995e-01,-1.592603e-01,2.748847e-01,-4.097211e-01,-6.064564e-01
75%,3.901882e-01,5.271497e-04,6.060636e-01,-4.097211e-01,9.283274e-01
max,2.236976e+01,2.946100e+01,1.599600e+00,1.042111e+01,2.463111e+00
